# Prompt Parsing
These are library test scripts, small code samples, and larger systems all designed around the prompt parsing and user interface aspect of this algorithm.

## Basic Snippets

### Basic `spaCy` Test
This script accepts user input and runs the `spaCy` model on it, tokenizing the input and printing out each token and its detected part of speech.

**Notes**
1. If the input is "dog red", red is parsed as a noun. But if the input is "red dog", red is parsed as an adjective

In [10]:
import spacy
from spacy.lang.en.examples import sentences 

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)
for token in doc:
    print(token.text, " | ", spacy.explain(token.pos_))

10 year old boy in red winter clothing
10  |  numeral
year  |  noun
old  |  adjective
boy  |  noun
in  |  adposition
red  |  adjective
winter  |  noun
clothing  |  noun


### Noun Filtering
This script accepts user input and after tokenizing it, prints out the detected nouns.

In [1]:
import spacy
from spacy.lang.en.examples import sentences 

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

nouns = []
for token in doc:
    if(spacy.explain(token.pos_) == "noun"):
        nouns.append(token.text)

for noun in nouns:
    print(noun)

10 year old boy in red winter clothing
year
boy
winter
clothing


### Adjective Filtering
This script accepts user input and after tokenizing it, prints out the detected adjectives.

In [11]:
import spacy
from spacy.lang.en.examples import sentences 

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

adjectives = []
for token in doc:
    if(spacy.explain(token.pos_) == "adjective"):
        adjectives.append(token.text)

for adj in adjectives:
    print(adj)

10 year old boy in red winter clothing
old
red


### Parsing Multiple Parts of Speech
This script accepts user input and after tokenizing it, prints out each word/token and its detected POS.

In [ ]:
import spacy

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

tokens = []

for token in doc:
    tokens.append({"text": token.text, "type": token.pos_})

for token in tokens:
    print(token["text"] + " - " + token["type"])

10 year old boy in red winter clothing
10- NUM
year- NOUN
old- ADJ
boy- NOUN
in- ADP
red- ADJ
winter- NOUN
clothing- NOUN


# Stage 1: Dependency Parsing
The first stage of this algorithm involves parsing the prompt the user gives. Previously, I looked at the basics of using the `spaCy` library and used it to parse the prompt into each word's part of speech.

This isn't particularly useful though because it doesn't tell us anything about each word. So we add dependency parsing which uses `spaCy` to figure out which word the current word is related to. This can help make relations and figure out what the prompt actually means.

## Basic Parsing
This script lists out each token and what word it relates to. 

In [6]:
import spacy

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)
print(doc.text)

tokens = []

for token in doc:
    tokens.append({"text": token.text, "type": spacy.explain(token.dep_), "dependency": token.head.text})

for token in tokens:
    print(f"{token["text"]} is a {token["type"]} and relates to {token["dependency"]}")

10 year old boy in red winter clothing lost in a dense forest
10 is a numeric modifier and relates to year
year is a noun phrase as adverbial modifier and relates to old
old is a adjectival modifier and relates to boy
boy is a nominal subject and relates to lost
in is a prepositional modifier and relates to boy
red is a adjectival modifier and relates to winter
winter is a compound and relates to clothing
clothing is a object of preposition and relates to in
lost is a root and relates to lost
in is a prepositional modifier and relates to lost
a is a determiner and relates to forest
dense is a adjectival modifier and relates to forest
forest is a object of preposition and relates to in


## Semantic Slots
Now we define certain slots that we would sort the tokens into for most prompts. This includes the main search object/subject, situations/verbs, physical descriptions, context/environmental clues, etc.

As you can see, this naive algorithm doesn't work quite as expected.

In [11]:
import spacy

text = input("Text to parse: ")

nlp = spacy.load("en_core_web_sm")
doc = nlp(text)

searchObject = []
situation = []
descriptions = []

print(doc.text)

for token in doc:
    if(spacy.explain(token.dep_) == "root"):
        searchObject.append(token.text)
    elif(spacy.explain(token.pos_) == "verb"):
        situation.append(token.text)
    elif(spacy.explain(token.pos_) == "adjective"):
        descriptions.append(token.text)

for obj in searchObject:
    print("OBJECT: ", obj)

for verb in situation:
    print("SITUATION: ", verb)

for adj in descriptions:
   print("DESCRIPTION: ", adj) 

10 year old boy in red winter clothing lost in the dense forest
OBJECT:  lost
DESCRIPTION:  old
DESCRIPTION:  red
DESCRIPTION:  dense


### A little help?
I wasn't sure where to go from here especially once I realized that not every prompt will have the same word types. Which means that even if I was able to figure out how to parse my test prompt, it would almost definitely not work for other sentence structures.

So I asked ChatGPT 5.2 the following prompt (actually a combination of multiple prompts, each with varying levels of success) and the following code block is the code that it gave me.

Note: I added a parsing function which wraps ChatGPT's code to allow for second prompt to be parsed at the same time to show how this algorithm doesn't work

> so i'm working on prompt parsing right now and i have a very basic snippet worked out that can take a prompt and spit out each word and its part of speech, what do i do now?

> help me understand what i need to do to parse this prompt: "10 year old boy in red winter clothing lost in the dense forest"

> give me code that can parse it exactly like that

In [33]:
import spacy

nlp = spacy.load("en_core_web_sm")

# This function's code is all from ChatGPT 
def parse():
    doc = nlp(prompt)

    result = {
        "target_entity": None,
        "age": None,
        "clothing": [],
        "state": None,
        "environment": {}
    }

    # --- 1. find state (ROOT verb or adjectival verb) ---
    for token in doc:
        if token.dep_ == "ROOT" and token.pos_ in ("VERB", "ADJ"):
            result["state"] = token.text
            state_token = token
            break

    # --- 2. find target entity (subject of state) ---
    for token in doc:
        if token.dep_ == "nsubj" and token.head == state_token:
            result["target_entity"] = token.text
            target = token
            break

    # --- 3. extract age ---
    for token in doc:
        if token.dep_ == "nummod" and token.head.text in ("year", "years"):
            # walk upward until boy
            head = token.head
            while head.head != head:
                if head.head == target:
                    result["age"] = token.text
                    break
                head = head.head

    # --- 4. extract clothing ---
    for token in doc:
        if token.text == "clothing" and token.dep_ == "pobj":
            for child in token.children:
                if child.dep_ in ("amod", "compound"):
                    result["clothing"].append(child.text)

    # --- 5. extract environment ---
    for token in doc:
        if token.dep_ == "pobj" and token.text == "forest":
            result["environment"]["type"] = token.text
            for child in token.children:
                if child.dep_ == "amod":
                    result["environment"]["descriptor"] = child.text

    print(result)

# Works
prompt = "10 year old boy in red winter clothing lost in the dense forest"
parse()

# Does NOT work
prompt = "small child wearing a blue jacket missing in the park"
parse()

{'target_entity': 'boy', 'age': '10', 'clothing': ['winter'], 'state': 'lost', 'environment': {'type': 'forest', 'descriptor': 'dense'}}
{'target_entity': None, 'age': None, 'clothing': [], 'state': None, 'environment': {}}
